# Accounts Receivable (AR) Synthetic Data Pipeline

> **Notebook:** `data_generator.ipynb`  
> **Stage:** 1 of 2 — Raw Data Generation  
> **Output:** `data/raw/Customers_Master.parquet` · `data/raw/AR_Invoices_950K.parquet` · `data/raw/Bank_Documents_Tracking.parquet`

This notebook demonstrates a high-performance data engineering approach to generating large-scale synthetic datasets (950K+ records) using vectorized operations with NumPy and Pandas, structured for a typical **Star Schema** (Fact and Dimension tables).

---
## ⚙️ Setup & Dependencies

Install required libraries if not already present.

In [1]:
!pip install pandas numpy faker pyarrow

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.0 MB 2.9 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.0 MB/s eta 0:00:00


---
## 1️⃣ Environment Setup

Importing core libraries for data manipulation, synthetic record generation, and performance tracking.  
We use `pathlib` for OS-agnostic paths and resolve output directories relative to the project root.

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
from pathlib import Path
import time

In [3]:
# ── Resolve project root & output directory ────────────────────────────────────
NOTEBOOK_DIR = Path.cwd()       # notebooks/ when run from Jupyter
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

# Auto-create raw data folder if it doesn't exist
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Start execution timer & initialise Faker ───────────────────────────────────
start_time = time.time()
fake = Faker()

---
## 📐 Step 1 — Define Dataset Scale

Set the target record counts for each table:

| Table | Records | Generation Method |
|-------|---------|-------------------|
| Customers (Dim) | 10,000 | Faker (row-by-row) |
| AR Invoices (Fact) | 950,000 | NumPy vectorized |
| Bank Documents | ~399K (70% of pending) | NumPy vectorized |

In [4]:
# ── Dataset scale configuration ────────────────────────────────────────────────
NUM_CUSTOMERS = 10000
NUM_INVOICES  = 950000

print("Generating Customers Dimension...")

Generating Customers Dimension...


---
## 👥 Step 2 — Generate Customers Dimension Table

The Customers dimension is relatively small (10K rows), so we use **Faker** for realistic company names.  
Payment terms are randomly drawn from a set of standard net-day options.

In [5]:
# ── Build Customers dimension with Faker (small table — row-by-row is fine) ────
customers_df = pd.DataFrame({
    'CustomerID'  : [f'C-{(i+1):05d}' for i in range(NUM_CUSTOMERS)],
    'CustomerName': [fake.company() for _ in range(NUM_CUSTOMERS)],
    'PaymentTerms': np.random.choice([30, 60, 90, 120], NUM_CUSTOMERS)
})

print("Generating Invoices Fact Table using NumPy (High Speed)...")

Generating Invoices Fact Table using NumPy (High Speed)...


---
## 🧾 Step 3 — Generate AR Invoices Fact Table

The 950K invoice table is built entirely with **vectorized NumPy operations** to maximise speed.  
Posting dates are drawn uniformly across a 2-year window ending today.

In [6]:
# ── Vectorized date generation across the last 2 years ────────────────────────
end_date        = datetime.today()
start_date      = end_date - timedelta(days=730)
date_range_days = (end_date - start_date).days

random_days_to_add = np.random.randint(0, date_range_days, NUM_INVOICES)

In [7]:
# ── Build invoices fact table ──────────────────────────────────────────────────
posting_dates = start_date + pd.to_timedelta(random_days_to_add, unit='D')

invoices_df = pd.DataFrame({
    'InvoiceID' : [f'INV-{(i+1):07d}' for i in range(NUM_INVOICES)],
    'CustomerID': np.random.choice(customers_df['CustomerID'], NUM_INVOICES),
    'PostingDate': posting_dates,
    'Amount'    : np.round(np.random.uniform(100.0, 150000.0, NUM_INVOICES), 2),
    'Status'    : np.random.choice(['Open', 'Partial', 'Cleared'], NUM_INVOICES, p=[0.4, 0.2, 0.4])
})

print("Generating Bank 2nd Set Tracking Table...")

Generating Bank 2nd Set Tracking Table...


---
## 🏦 Step 4 — Generate Bank Documents Tracking Table

We simulate bank submission records for **70%** of all pending (Open or Partial) invoices.  
Each document is assigned a submission date 5–30 days after the invoice posting date.

In [8]:
# ── Filter pending invoices and build bank-tracking table ──────────────────────
pending_invoices = invoices_df[invoices_df['Status'].isin(['Open', 'Partial'])]['InvoiceID'].values
NUM_BANK_DOCS    = int(len(pending_invoices) * 0.7)  # assume 70% were submitted to the bank

bank_docs_df = pd.DataFrame({
    'DocID'             : [f'DOC-{(i+1):06d}' for i in range(NUM_BANK_DOCS)],
    'InvoiceID'         : np.random.choice(pending_invoices, NUM_BANK_DOCS, replace=False),
    'BankSubmissionDate': posting_dates[:NUM_BANK_DOCS] + pd.to_timedelta(np.random.randint(5, 30, NUM_BANK_DOCS), unit='D'),
    'DocStatus'         : np.random.choice(['Under Review', 'Accepted', 'Rejected'], NUM_BANK_DOCS, p=[0.6, 0.3, 0.1])
})

---
## 💾 Step 5 — Export to Parquet

All three tables are saved to `data/raw/` in **Parquet format** using `pyarrow` — ~77 % smaller than CSV, with full schema/type preservation and significantly faster reads for the ETL engine.


In [9]:
# ── Export all tables to the raw data directory (Parquet format) —————————————
# التصدير السابق بصيغة CSV (موقوف)
# customers_df.to_csv(RAW_DATA_DIR / 'Customers_Master.csv',        index=False)
# invoices_df .to_csv(RAW_DATA_DIR / 'AR_Invoices_950K.csv',        index=False)
# bank_docs_df.to_csv(RAW_DATA_DIR / 'Bank_Documents_Tracking.csv', index=False)

# التصدير الجديد بصيغة Parquet
print('Exporting to Parquet...')
customers_df.to_parquet(RAW_DATA_DIR / 'Customers_Master.parquet',        engine='pyarrow', index=False)
print('  ✅ Customers_Master.parquet')
invoices_df .to_parquet(RAW_DATA_DIR / 'AR_Invoices_950K.parquet',        engine='pyarrow', index=False)
print('  ✅ AR_Invoices_950K.parquet')
bank_docs_df.to_parquet(RAW_DATA_DIR / 'Bank_Documents_Tracking.parquet', engine='pyarrow', index=False)
print('  ✅ Bank_Documents_Tracking.parquet')

print(f"Done! Total execution time: {round(time.time() - start_time, 2)} seconds.")

Exporting to Parquet...
  ✅ Customers_Master.parquet
  ✅ AR_Invoices_950K.parquet
  ✅ Bank_Documents_Tracking.parquet
Done! Total execution time: 10.76 seconds.


---
## ✅ Step 6 — Data Quality Assurance (DQA)

As a best practice in data engineering, we verify the integrity of the generated data before downstream consumption:

- **Row counts** match the configured targets
- **Referential integrity** — every `CustomerID` in the invoices table exists in the customers dimension
- **Statistical preview** — date range and amount distribution look plausible

In [10]:
# ── Data Quality Assurance checks ─────────────────────────────────────────────
print("--- Data Summary ---")
print(f"Invoices Count: {len(invoices_df):,}")
print(f"Customers Count: {len(customers_df):,}")
print(f"Bank Documents Count: {len(bank_docs_df):,}")

# Validate referential integrity
missing_customers = invoices_df[~invoices_df['CustomerID'].isin(customers_df['CustomerID'])]
print(f"Referential Integrity Issues: {len(missing_customers)}")

# Preview data
display(invoices_df.head())
display(invoices_df.describe())


--- Data Summary ---
Invoices Count: 950,000
Customers Count: 10,000
Bank Documents Count: 399,219
Referential Integrity Issues: 0


,InvoiceID,CustomerID,PostingDate,Amount,Status
0,INV-0000001,C-01221,2024-10-05 18:42:03.727963,134033.68,Cleared
1,INV-0000002,C-07499,2026-03-06 18:42:03.727963,72333.33,Cleared
2,INV-0000003,C-02056,2024-12-16 18:42:03.727963,67937.34,Cleared
3,INV-0000004,C-05863,2024-12-24 18:42:03.727963,81729.98,Cleared
4,INV-0000005,C-04854,2024-06-21 18:42:03.727963,9035.96,Cleared


,PostingDate,Amount
count,950000,950000.000000
mean,2025-04-20 03:20:36.731752448,75001.579183
min,2024-04-20 18:42:03.727963,100.110000
25%,2024-10-19 18:42:03.727962880,37544.650000
50%,2025-04-19 18:42:03.727962880,74959.140000
75%,2025-10-19 18:42:03.727962880,112515.307500
max,2026-04-19 18:42:03.727963,149999.900000
std,NaN,43285.640499
